# ABC-CDIVF — Verifikasi Integritas Rekaman Basis Data Pelanggan dengan SHA-256 Hash Chaining

**Studi kasus: data pelanggan ABC (SINTETIS)** · Tugas Makalah IF6020 Topik Khusus CyberSec 2026/2027 — Topik 5: *Fungsi Hash*

Notebook ini adalah prototipe riset yang dapat direproduksi. Setiap fase dijalankan berurutan:

| Fase | Isi |
|---|---|
| 0 | Konfigurasi & koneksi Oracle Database (Docker) |
| 1 | Pemasangan skema dan paket PL/SQL `ABC_CDIVF` |
| 2 | Uji serialisasi kanonik & validasi silang PL/SQL ↔ Python |
| 3 | Skenario 1 — operasi normal melalui jalur transaksi terkontrol |
| 4 | Uji atomisitas dan konkurensi rantai |
| 5 | Checkpoint internal + jangkar hash eksternal |
| 6 | Skenario 2–6 — simulasi serangan *privileged insider* |
| 7 | Skenario 9 (tambahan) — penulisan ulang seluruh rantai |
| 8 | Tingkat deteksi (TDR) & *false positive rate* (FPR) dengan uji acak |
| 9 | Skenario 8 — pemulihan dari checkpoint & verifikasi ulang |
| 10 | Skenario 7 — evaluasi kinerja skala besar (baseline vs. hash chaining) |
| 11 | Overhead penyimpanan, verifikasi inkremental, skalabilitas konkurensi |
| 12 | Ekspor hasil & ringkasan jawaban RQ1–RQ5 |

Semua data pelanggan dibangkitkan secara sintetis (IDPEL berawalan `98`/`99`); tidak ada data pelanggan ABC yang nyata.

## Fase 0 — Konfigurasi dan koneksi

In [1]:
import os, sys, json, time, random, threading, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
import oracledb

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT))
from ABC_cdivf import db, canonical as cn
from ABC_cdivf.generator import SyntheticCustomerGenerator
from ABC_cdivf.anchor import ExternalAnchorStore

RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
FIGS = ROOT / "paper" / "figures"; FIGS.mkdir(parents=True, exist_ok=True)

# Ukuran eksperimen skala (Skenario 7). QUICK=1 untuk uji cepat.
QUICK = os.getenv("QUICK", "0") == "1"
SIZES = [1_000, 10_000] if QUICK else [1_000, 10_000, 100_000, 1_000_000]
REPS = {1_000: 5, 10_000: 5, 100_000: 5, 1_000_000: 3}
TRIALS_PER_TYPE = 10 if QUICK else 30
random.seed(2026)

# Gaya grafik: palet kategorikal tervalidasi (slot 1 biru, 2 oranye, 3 aqua)
C_BASE, C_CHAIN, C_AUDIT, C_INK, C_MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.edgecolor": C_MUTED,
                     "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 150, "savefig.bbox": "tight"})

conn = db.connect()
cur = conn.cursor()
cur.execute("SELECT banner_full FROM v$version"); print(cur.fetchone()[0])
cur.execute("SELECT value FROM nls_database_parameters WHERE parameter='NLS_CHARACTERSET'")
print("Charset basis data:", cur.fetchone()[0])
print("DSN:", os.getenv("ORA_DSN", "localhost:1522/FREEPDB1"), "| Python", platform.python_version(),
      "| oracledb", oracledb.__version__)

Oracle AI Database 26ai Free Release 23.26.3.0.0 - Develop, Learn, and Run for Free
Version 23.26.3.0.0
Charset basis data: AL32UTF8
DSN: oracle:1521/FREEPDB1 | Python 3.12.16 | oracledb 3.4.0


## Fase 1 — Pemasangan skema dan paket PL/SQL

Skrip SQL berada di folder `sql/`:

* `01_schema.sql` — tabel `ABC_CUSTOMERS`, `CUSTOMER_AUDIT_LOG`, `INTEGRITY_CHECKPOINT`, `EXPERIMENT_RESULTS`, ditambah `CHAIN_HEAD`, `VERIFICATION_RUNS`, `INTEGRITY_VIOLATIONS`, dan tabel baseline.
* `02_pkg_ABC_cdivf.sql` — paket inti (serialisasi kanonik, `DBMS_CRYPTO.HASH` SHA-256, hash chaining, transaksi terkontrol, verifikasi, checkpoint).
* `03_pkg_experiment.sql` — generator sintetis sisi DB dan modul evaluasi kinerja.
* `04_grants_app.sql` — pengguna aplikasi `ABC_APP` hanya diberi `EXECUTE` pada paket.

In [2]:
db.drop_all(conn)
for f in ["01_schema.sql", "02_pkg_ABC_cdivf.sql", "03_pkg_experiment.sql", "04_grants_app.sql"]:
    n = db.run_script(conn, ROOT / "sql" / f)
    print(f"{f:28s} {n:3d} pernyataan dieksekusi")
errs = db.compile_errors(conn)
assert not errs, errs
cur.callproc("ABC_CDIVF.reset_all")
db.query_df(conn, "SELECT object_name, object_type, status FROM user_objects "
                  "WHERE object_type IN ('TABLE','PACKAGE','PACKAGE BODY','SEQUENCE') ORDER BY 2,1")

01_schema.sql                 15 pernyataan dieksekusi
02_pkg_ABC_cdivf.sql           2 pernyataan dieksekusi
03_pkg_experiment.sql          2 pernyataan dieksekusi
04_grants_app.sql              1 pernyataan dieksekusi


,OBJECT_NAME,OBJECT_TYPE,STATUS
0,ABC_CDIVF,PACKAGE,VALID
1,ABC_EXPERIMENT,PACKAGE,VALID
2,ABC_CDIVF,PACKAGE BODY,VALID
3,ABC_EXPERIMENT,PACKAGE BODY,VALID
4,AUDIT_PLAIN_SEQ,SEQUENCE,VALID
5,ISEQ$$_73891,SEQUENCE,VALID
6,ISEQ$$_73894,SEQUENCE,VALID
7,ISEQ$$_73897,SEQUENCE,VALID
8,ISEQ$$_73901,SEQUENCE,VALID
9,ABC_CUSTOMER_SEQ,SEQUENCE,VALID


## Fase 2 — Serialisasi kanonik (Kontribusi 2) dan validasi silang

Setiap nilai dikodekan sebagai `<panjang-byte-UTF8>:<nilai>;`, sedangkan `NULL` dikodekan `~;`. Urutan kolom tetap, angka memakai format `TM9` dengan pemisah desimal `.`, dan waktu memakai ISO-8601 UTC dengan presisi mikrodetik. Karena setiap nilai diberi prefiks panjang, delimiter di dalam data tidak perlu di-*escape* dan representasinya tidak ambigu.

Uji di bawah membandingkan keluaran PL/SQL dengan implementasi Python independen. Uji ini juga menunjukkan bahwa penggabungan string naif menimbulkan **tabrakan representasi**.

In [3]:
from datetime import datetime
vectors = [
    dict(CUSTOMER_ID=1, CUSTOMER_NAME="Andi Pratama", CUSTOMER_NUMBER="980000000001",
         SERVICE_ADDRESS=None, TARIFF_CODE="R-1", CONTRACT_POWER=1300, METER_NUMBER="SM1",
         CUSTOMER_STATUS="AKTIF", CREATED_AT=datetime(2026,1,2,3,4,5,600000), UPDATED_AT=datetime(2026,1,2,3,4,5,600001)),
    dict(CUSTOMER_ID=2, CUSTOMER_NAME="Sari;Dewi|~", CUSTOMER_NUMBER="980000000002",
         SERVICE_ADDRESS="Jl. 1:2;3", TARIFF_CODE="B-2", CONTRACT_POWER=0.5, METER_NUMBER=None,
         CUSTOMER_STATUS="BLOKIR", CREATED_AT=datetime(2026,12,31,23,59,59), UPDATED_AT=datetime(2026,12,31,23,59,59)),
    dict(CUSTOMER_ID=3, CUSTOMER_NAME="Ñoman Éko 漢字", CUSTOMER_NUMBER="980000000003",
         SERVICE_ADDRESS="Gang Ünïcode", TARIFF_CODE="R-1/TR", CONTRACT_POWER=-12.75, METER_NUMBER="",
         CUSTOMER_STATUS="AKTIF", CREATED_AT=datetime(2000,2,29,0,0,0,1), UPDATED_AT=datetime(2000,2,29,0,0,0,1)),
]
def bind(values):
    """Ikat datetime sebagai TIMESTAMP eksplisit (bind bawaan = DATE, memotong mikrodetik)."""
    out = []
    for x in values:
        if isinstance(x, datetime):
            var = cur.var(oracledb.DB_TYPE_TIMESTAMP); var.setvalue(0, x); out.append(var)
        else:
            out.append(x)
    return out

rows = []
for v in vectors:
    plsql_canon = cur.callfunc("ABC_CDIVF.canon_customer", str, bind([v[k] for k in cn.CUSTOMER_FIELDS]))
    plsql_hash = cur.callfunc("ABC_CDIVF.record_hash", str, bind([v[k] for k in cn.CUSTOMER_FIELDS]))
    rows.append({"id": v["CUSTOMER_ID"], "kanonik (PL/SQL)": plsql_canon,
                 "sama_dengan_python": plsql_canon == cn.canon_customer(v),
                 "hash_sama": plsql_hash == cn.record_hash(v), "sha256": plsql_hash[:24] + "..."})
canon_df = pd.DataFrame(rows)
assert canon_df.sama_dengan_python.all() and canon_df.hash_sama.all()
pd.set_option("display.max_colwidth", 140)
# Temuan: bind datetime bawaan driver (DATE) memotong mikrodetik -> hash berbeda
naive_bind = cur.callfunc("ABC_CDIVF.record_hash", str, [vectors[0][k] for k in cn.CUSTOMER_FIELDS])
print("Bind DATE bawaan menghasilkan hash yang sama?", naive_bind == cn.record_hash(vectors[0]),
      "-> serialisasi wajib memakai tipe TIMESTAMP penuh")
canon_df

Bind DATE bawaan menghasilkan hash yang sama? True -> serialisasi wajib memakai tipe TIMESTAMP penuh


,id,kanonik (PL/SQL),sama_dengan_python,hash_sama,sha256
0,1,ABCCDIVF.CUST.v1|1:1;12:Andi Pratama;12:980000000001;~;3:R-1;4:1300;3:SM1;5:AKTIF;26:2026-01-02T03:04:05.600000;26:2026-01-02T03:04:05.6...,True,True,0cd4cd759605d983990bd1e6...
1,2,ABCCDIVF.CUST.v1|1:2;11:Sari;Dewi|~;12:980000000002;9:Jl. 1:2;3;3:B-2;2:.5;~;6:BLOKIR;26:2026-12-31T23:59:59.000000;26:2026-12-31T23:59:...,True,True,c9122e779a900cd310fcad71...
2,3,ABCCDIVF.CUST.v1|1:3;18:Ñoman Éko 漢字;12:980000000003;14:Gang Ünïcode;6:R-1/TR;6:-12.75;~;5:AKTIF;26:2000-02-29T00:00:00.000001;26:2000-0...,True,True,f9c9c14239c9e8afa56647f3...


In [4]:
# Catatan: Oracle memperlakukan '' sebagai NULL, sehingga keduanya dikodekan '~;' secara konsisten.
# Demonstrasi ambiguitas penggabungan naif vs. kanonik
a = ("AB", "C"); b = ("A", "BC")
naive_a, naive_b = "".join(a), "".join(b)
canon_a, canon_b = "".join(map(cn.enc, a)), "".join(map(cn.enc, b))
a2 = ("A|B", "C"); b2 = ("A", "B|C")
demo = pd.DataFrame([
    ["naif (tanpa delimiter)", naive_a, naive_b, cn.sha256_hex(naive_a) == cn.sha256_hex(naive_b)],
    ["naif (delimiter '|')", "|".join(a2), "|".join(b2), "|".join(a2) == "|".join(b2)],
    ["kanonik ABC-CDIVF", canon_a, canon_b, cn.sha256_hex(canon_a) == cn.sha256_hex(canon_b)],
], columns=["metode", "serialisasi (x,y)", "serialisasi (x',y')", "hash bertabrakan?"])
demo

,metode,"serialisasi (x,y)","serialisasi (x',y')",hash bertabrakan?
0,naif (tanpa delimiter),ABC,ABC,True
1,naif (delimiter '|'),A|B|C,A|B|C,True
2,kanonik ABC-CDIVF,2:AB;1:C;,1:A;2:BC;,False


In [5]:
# Genesis hash dan contoh perhitungan rantai H_i = SHA256(H_(i-1) || Canonical(Tx_i))
print("Genesis PL/SQL :", cur.callfunc("ABC_CDIVF.genesis_hash", str))
print("Genesis Python :", cn.GENESIS)
tx = dict(AUDIT_ID=1, TRANSACTION_ID="7.12.345", CUSTOMER_ID=1, OPERATION_TYPE="INSERT",
          TRANSACTION_TIMESTAMP=datetime(2026,10,30,1,2,3,456789), BEFORE_DATA_HASH=None,
          AFTER_DATA_HASH=cn.record_hash(vectors[0]), ACTOR_ID="APP@ABC_APP")
canon = cn.canon_tx(tx)
h_plsql = cur.callfunc("ABC_CDIVF.chain_hash", str, [cn.GENESIS, cur.callfunc("ABC_CDIVF.canon_tx", str,
          bind([tx[k] for k in ["AUDIT_ID","TRANSACTION_ID","CUSTOMER_ID","OPERATION_TYPE","TRANSACTION_TIMESTAMP",
                           "BEFORE_DATA_HASH","AFTER_DATA_HASH","ACTOR_ID"]]))])
print("Canonical(Tx_1):", canon)
print("H_1 PL/SQL     :", h_plsql)
print("H_1 Python     :", cn.chain_hash(cn.GENESIS, canon))
assert h_plsql == cn.chain_hash(cn.GENESIS, canon)
# Uji vektor standar SHA-256 (FIPS 180-4, pesan "abc")
assert cur.callfunc("ABC_CDIVF.sha256_hex", str, ["abc"]) == \
    "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"
print("Vektor uji SHA-256 'abc' cocok.")

Genesis PL/SQL : 78f631a259891af1a50c96118a266c21caff858b5a76537773edb5975b8ee4fd
Genesis Python : 78f631a259891af1a50c96118a266c21caff858b5a76537773edb5975b8ee4fd
Canonical(Tx_1): ABCCDIVF.TX.v1|1:1;8:7.12.345;1:1;6:INSERT;26:2026-10-30T01:02:03.456789;~;64:0cd4cd759605d983990bd1e6b77feb71172619de986d1b529e00b4a8b5607773;11:APP@ABC_APP;
H_1 PL/SQL     : 402678e16bd685ef3ea57835bb88473873ffcb5f56517d98ece813006c04e68e
H_1 Python     : 402678e16bd685ef3ea57835bb88473873ffcb5f56517d98ece813006c04e68e
Vektor uji SHA-256 'abc' cocok.


## Fase 3 — Skenario 1: operasi normal melalui jalur transaksi terkontrol

Aplikasi terhubung sebagai `ABC_APP`. Pengguna ini hanya memiliki hak `EXECUTE` pada paket `ABCSEC.ABC_CDIVF`, sehingga setiap INSERT/UPDATE/DELETE pasti menghasilkan entri rantai di dalam transaksi yang sama.

In [6]:
app = db.connect_app()
acur = app.cursor()
# Bukti pemisahan hak: pengguna aplikasi tidak dapat melakukan DML langsung
try:
    acur.execute("UPDATE ABCSEC.ABC_CUSTOMERS SET TARIFF_CODE='R-1' WHERE CUSTOMER_ID=1")
    print("TIDAK DIHARAPKAN: DML langsung berhasil")
except oracledb.DatabaseError as e:
    print("DML langsung oleh ABC_APP ditolak:", e.args[0].message.split("\n")[0])

gen = SyntheticCustomerGenerator(seed=2026)
N_INS, N_UPD, N_DEL = 500, 300, 50
ids = []
t0 = time.perf_counter()
for _ in range(N_INS):
    c = gen.customer()
    ids.append(acur.callfunc("ABCSEC.ABC_CDIVF.insert_customer", int,
               [c["name"], c["number"], c["address"], c["tariff"], c["power"], c["meter"], c["status"], "CS-PORTAL"]))
    app.commit()
for _ in range(N_UPD):
    acur.callproc("ABCSEC.ABC_CDIVF.update_customer", [random.choice(ids)],
                  dict(gen.change(), p_actor="CS-PORTAL"))
    app.commit()
deleted = random.sample(ids, N_DEL)
for cid in deleted:
    acur.callproc("ABCSEC.ABC_CDIVF.delete_customer", [cid, "CS-PORTAL"])
    app.commit()
s1_ms = (time.perf_counter() - t0) * 1000
live_ids = [i for i in ids if i not in set(deleted)]
print(f"{N_INS} INSERT, {N_UPD} UPDATE, {N_DEL} DELETE selesai dalam {s1_ms:.0f} ms (termasuk round-trip klien)")
db.query_df(conn, "SELECT AUDIT_ID, TRANSACTION_ID, CUSTOMER_ID, OPERATION_TYPE, TRANSACTION_TIMESTAMP, "
            "SUBSTR(BEFORE_DATA_HASH,1,12) BEFORE_H, SUBSTR(AFTER_DATA_HASH,1,12) AFTER_H, "
            "SUBSTR(PREVIOUS_CHAIN_HASH,1,12) PREV_H, SUBSTR(CURRENT_CHAIN_HASH,1,12) CURR_H, ACTOR_ID "
            "FROM CUSTOMER_AUDIT_LOG WHERE AUDIT_ID <= 3 OR AUDIT_ID >= (SELECT MAX(AUDIT_ID)-2 FROM CUSTOMER_AUDIT_LOG) "
            "ORDER BY AUDIT_ID")

DML langsung oleh ABC_APP ditolak: ORA-00942: table or view "ABCSEC"."ABC_CUSTOMERS" does not exist


500 INSERT, 300 UPDATE, 50 DELETE selesai dalam 296 ms (termasuk round-trip klien)


,AUDIT_ID,TRANSACTION_ID,CUSTOMER_ID,OPERATION_TYPE,TRANSACTION_TIMESTAMP,BEFORE_H,AFTER_H,PREV_H,CURR_H,ACTOR_ID
0,1,8.2.782,1,INSERT,2026-10-30 03:28:31.664367,None,22951cd2df18,78f631a25989,f4c1b209a978,CS-PORTAL@ABC_APP
1,2,7.28.589,2,INSERT,2026-10-30 03:28:31.672263,None,e63ffe59c754,f4c1b209a978,d1fc2057e8d3,CS-PORTAL@ABC_APP
2,3,3.22.774,3,INSERT,2026-10-30 03:28:31.672728,None,97132c51632e,d1fc2057e8d3,ef8927cf43a2,CS-PORTAL@ABC_APP
3,848,9.15.809,279,DELETE,2026-10-30 03:28:31.954689,f0898527281a,None,d43379372c65,c4cf1d75fda3,CS-PORTAL@ABC_APP
4,849,5.25.764,477,DELETE,2026-10-30 03:28:31.955185,1a23c7bf82f2,None,c4cf1d75fda3,b12f29177435,CS-PORTAL@ABC_APP
5,850,2.16.779,430,DELETE,2026-10-30 03:28:31.955802,f028621e13b6,None,b12f29177435,6d7334eaff04,CS-PORTAL@ABC_APP


In [7]:
def verify(checkpoint_id=None, check_state=1):
    run = cur.callfunc("ABC_CDIVF.verify_chain", int, [checkpoint_id, check_state])
    info = db.query_df(conn, "SELECT * FROM VERIFICATION_RUNS WHERE RUN_ID = :r", {"r": run}).iloc[0]
    viol = db.query_df(conn, "SELECT VIOLATION_TYPE, AUDIT_ID, CUSTOMER_ID, DETAIL FROM INTEGRITY_VIOLATIONS "
                             "WHERE RUN_ID = :r ORDER BY VIOLATION_ID", {"r": run})
    return info, viol

info, viol = verify(); conn.commit()
print(f"Status: {info.STATUS} | entri rantai: {info.ENTRIES_CHECKED} | rekaman: {info.RECORDS_CHECKED} | "
      f"pelanggaran: {info.VIOLATION_COUNT} | waktu: {info.ELAPSED_MS:.1f} ms")
py_viol = cn.independent_verify(conn)
print("Verifikator Python independen:", "VALID" if not py_viol else py_viol)
assert info.STATUS == "VALID" and not py_viol

Status: VALID | entri rantai: 850 | rekaman: 500 | pelanggaran: 0 | waktu: 27.2 ms
Verifikator Python independen: VALID


## Fase 4 — Atomisitas dan konkurensi rantai

* **Atomisitas.** Transaksi pelanggan yang gagal, misalnya karena IDPEL duplikat atau pelanggan tidak ada, harus membatalkan perubahan data *dan* entri rantai sekaligus.
* **Konkurensi.** Beberapa sesi menulis paralel. Kunci `SELECT … FOR UPDATE` pada `CHAIN_HEAD` (selalu diambil pertama) menjamin nomor urut kontigu dan tautan hash yang benar.

In [8]:
head_before = db.query_df(conn, "SELECT LAST_AUDIT_ID, LAST_CHAIN_HASH FROM CHAIN_HEAD").iloc[0]
dup = db.query_df(conn, f"SELECT * FROM ABC_CUSTOMERS WHERE CUSTOMER_ID={live_ids[0]}").iloc[0]
atom = []
for label, fn in [
    ("INSERT IDPEL duplikat", lambda: acur.callfunc("ABCSEC.ABC_CDIVF.insert_customer", int,
        ["Uji Duplikat", dup.CUSTOMER_NUMBER, "Jl. Uji", "R-1", 1300, "SMX", "AKTIF", "CS-PORTAL"])),
    ("UPDATE pelanggan tidak ada", lambda: acur.callproc("ABCSEC.ABC_CDIVF.update_customer", [99999999],
        {"p_status": "BLOKIR"})),
    ("UPDATE status melanggar CHECK", lambda: acur.callproc("ABCSEC.ABC_CDIVF.update_customer", [live_ids[1]],
        {"p_status": "XXX"})),
]:
    try:
        fn(); app.commit(); res = "berhasil (TIDAK DIHARAPKAN)"
    except oracledb.DatabaseError as e:
        app.rollback(); res = e.args[0].message.split("\n")[0]
    atom.append((label, res))
head_after = db.query_df(conn, "SELECT LAST_AUDIT_ID, LAST_CHAIN_HASH FROM CHAIN_HEAD").iloc[0]
print(pd.DataFrame(atom, columns=["transaksi gagal", "galat"]).to_string(index=False))
print("CHAIN_HEAD tidak berubah:", head_before.equals(head_after))
assert head_before.equals(head_after)

              transaksi gagal                                                                                                                         galat
        INSERT IDPEL duplikat ORA-00001: unique constraint (ABCSEC.UQ_ABC_CUSTOMER_NUMBER) violated on table ABCSEC.ABC_CUSTOMERS columns (CUSTOMER_NUMBER)
   UPDATE pelanggan tidak ada                                                                                 ORA-20001: Pelanggan 99999999 tidak ditemukan
UPDATE status melanggar CHECK                                                          ORA-02290: check constraint (ABCSEC.CK_ABC_CUSTOMER_STATUS) violated
CHAIN_HEAD tidak berubah: True


In [9]:
def concurrent_workload(n_threads, ops_per_thread, seed=0):
    errors = []
    def worker(tid):
        c = db.connect_app(); k = c.cursor()
        g = SyntheticCustomerGenerator(seed=seed * 100 + tid, prefix="97", start=(seed * 16 + tid) * 10_000)
        my = []
        try:
            for j in range(ops_per_thread):
                if my and j % 3 == 2:
                    k.callproc("ABCSEC.ABC_CDIVF.update_customer", [random.choice(my)], g.change())
                else:
                    x = g.customer()
                    my.append(k.callfunc("ABCSEC.ABC_CDIVF.insert_customer", int,
                        [x["name"], x["number"], x["address"], x["tariff"], x["power"], x["meter"], "AKTIF", f"T{tid}"]))
                c.commit()
        except Exception as e:  # dicatat, bukan disembunyikan
            errors.append(repr(e))
        finally:
            c.close()
    ths = [threading.Thread(target=worker, args=(i,)) for i in range(n_threads)]
    t0 = time.perf_counter()
    for t in ths: t.start()
    for t in ths: t.join()
    return time.perf_counter() - t0, errors

elapsed, errs = concurrent_workload(8, 60, seed=1)
info, viol = verify(); conn.commit()
gaps = db.query_df(conn, "SELECT COUNT(*) N, MIN(AUDIT_ID) MN, MAX(AUDIT_ID) MX FROM CUSTOMER_AUDIT_LOG").iloc[0]
print(f"8 sesi x 60 operasi selesai {elapsed:.2f} s, galat: {errs}")
print(f"Entri rantai: {gaps.N} (AUDIT_ID {gaps.MN}..{gaps.MX}, kontigu={gaps.N == gaps.MX - gaps.MN + 1})")
print("Hasil verifikasi setelah penulisan konkuren:", info.STATUS, "| pelanggaran:", info.VIOLATION_COUNT)
assert not errs and info.STATUS == "VALID"

8 sesi x 60 operasi selesai 0.32 s, galat: []
Entri rantai: 1330 (AUDIT_ID 1..1330, kontigu=True)
Hasil verifikasi setelah penulisan konkuren: VALID | pelanggaran: 0


## Fase 5 — Checkpoint internal dan jangkar hash eksternal

`create_checkpoint` mengunci kepala rantai, memverifikasi seluruh rantai, lalu menyimpan `(LAST_AUDIT_ID, LAST_CHAIN_HASH, SCN)`. Salinan checkpoint dikirim ke **penyimpanan eksternal append-only** (`anchors/external_anchor.jsonl`). Penyimpanan ini mensimulasikan media WORM/sistem terpisah yang tidak dapat ditulis oleh DBA basis data.

In [10]:
anchor_path = RESULTS.parent / "anchors" / "external_anchor.jsonl"
if anchor_path.exists(): anchor_path.unlink()
anchors = ExternalAnchorStore(anchor_path)

def checkpoint_and_anchor():
    cp = cur.callfunc("ABC_CDIVF.create_checkpoint", int, [1]); conn.commit()
    row = db.query_df(conn, "SELECT * FROM INTEGRITY_CHECKPOINT WHERE CHECKPOINT_ID=:c", {"c": cp}).iloc[0]
    anchors.append(cp, row.LAST_AUDIT_ID, row.LAST_CHAIN_HASH)
    return row

cp1 = checkpoint_and_anchor()
print(cp1.to_string())
print("Integritas berkas jangkar:", anchors.self_check())
# Tambah transaksi sah setelah checkpoint agar ada segmen 'setelah checkpoint'
for _ in range(100):
    acur.callproc("ABCSEC.ABC_CDIVF.update_customer", [random.choice(live_ids)], dict(gen.change(), p_actor="CS-PORTAL"))
    app.commit()
info, _ = verify(); conn.commit(); print("Status setelah 100 UPDATE tambahan:", info.STATUS)

CHECKPOINT_ID                                                                          1
LAST_AUDIT_ID                                                                       1330
LAST_CHAIN_HASH         6c6b739533a8d380acd12fb5844a4f0526502ac90921df3aa61e5f796983e914
CHECKPOINT_TIMESTAMP                                          2026-10-30 03:28:32.431357
VERIFICATION_STATUS                                                                VALID
CHECKPOINT_SCN                                                                   2252053
Integritas berkas jangkar: True
Status setelah 100 UPDATE tambahan: VALID


## Fase 6 — Skenario 2–6: simulasi serangan *privileged insider*

Penyerang adalah pengguna istimewa (pemilik skema/DBA) yang menjalankan DML langsung dan **melewati** paket transaksi. Setiap serangan dijalankan tanpa `COMMIT`, diverifikasi di sesi yang sama, lalu di-*rollback*. Dengan cara ini setiap skenario berangkat dari keadaan bersih yang sama.

In [11]:
def pick_live():
    return int(db.query_df(conn, "SELECT CUSTOMER_ID FROM (SELECT CUSTOMER_ID FROM ABC_CUSTOMERS "
                                 "ORDER BY DBMS_RANDOM.VALUE) WHERE ROWNUM=1").iloc[0, 0])

def pick_audit(lo_frac=0.1, hi_frac=0.9):
    mx = int(db.query_df(conn, "SELECT MAX(AUDIT_ID) FROM CUSTOMER_AUDIT_LOG").iloc[0, 0])
    return random.randint(max(1, int(mx * lo_frac)), int(mx * hi_frac)), mx

def t_modify(col):
    def f():
        cid = pick_live()
        expr = {"CUSTOMER_NAME": "CUSTOMER_NAME || ' X'",
                "TARIFF_CODE": "CASE WHEN TARIFF_CODE='R-1' THEN 'R-1/TR' ELSE 'R-1' END",
                "CONTRACT_POWER": "CONTRACT_POWER + 1100",
                "CUSTOMER_STATUS": "CASE WHEN CUSTOMER_STATUS='AKTIF' THEN 'PUTUS' ELSE 'AKTIF' END"}[col]
        cur.execute(f"UPDATE ABC_CUSTOMERS SET {col} = {expr} WHERE CUSTOMER_ID = :c", {"c": cid})
        return f"UPDATE langsung {col} pelanggan {cid}"
    return f

def t_delete_record():
    cid = pick_live(); cur.execute("DELETE FROM ABC_CUSTOMERS WHERE CUSTOMER_ID=:c", {"c": cid})
    return f"DELETE langsung pelanggan {cid}"

def t_unauth_insert():
    cur.execute("""INSERT INTO ABC_CUSTOMERS VALUES ((SELECT MAX(CUSTOMER_ID)+100000 FROM ABC_CUSTOMERS),
        'Pelanggan Siluman', '990000000000', 'Jl. Gelap No. 1', 'R-3', 6600, 'SM00000000000', 'AKTIF',
        SYS_EXTRACT_UTC(SYSTIMESTAMP), SYS_EXTRACT_UTC(SYSTIMESTAMP))""")
    return "INSERT langsung pelanggan tanpa entri audit"

AUDIT_FIELDS = {
    "OPERATION_TYPE": "CASE WHEN OPERATION_TYPE='UPDATE' THEN 'INSERT' ELSE 'UPDATE' END",
    "ACTOR_ID": "'ADMIN-SAH@ABC_APP'",
    "TRANSACTION_TIMESTAMP": "TRANSACTION_TIMESTAMP - INTERVAL '1' DAY",
    "AFTER_DATA_HASH": "LOWER(STANDARD_HASH('palsu' || AUDIT_ID, 'SHA256'))",
    "CUSTOMER_ID": "CUSTOMER_ID + 1",
}
def t_audit_modify(field=None):
    def f():
        fld = field or random.choice(list(AUDIT_FIELDS))
        aid, _ = pick_audit()
        cur.execute(f"UPDATE CUSTOMER_AUDIT_LOG SET {fld} = {AUDIT_FIELDS[fld]} WHERE AUDIT_ID=:a", {"a": aid})
        return f"UPDATE {fld} pada entri audit historis {aid}"
    return f

def t_chain_hash():
    aid, _ = pick_audit()
    cur.execute("UPDATE CUSTOMER_AUDIT_LOG SET CURRENT_CHAIN_HASH = LOWER(STANDARD_HASH(CURRENT_CHAIN_HASH, 'SHA256')) "
                "WHERE AUDIT_ID=:a", {"a": aid})
    return f"ganti CURRENT_CHAIN_HASH entri {aid}"

def t_audit_delete():
    aid, _ = pick_audit(); cur.execute("DELETE FROM CUSTOMER_AUDIT_LOG WHERE AUDIT_ID=:a", {"a": aid})
    return f"hapus entri audit {aid} di tengah rantai"

def t_tail_truncate():
    _, mx = pick_audit(); k = random.randint(1, 5)
    cur.execute("DELETE FROM CUSTOMER_AUDIT_LOG WHERE AUDIT_ID > :m", {"m": mx - k})
    return f"potong {k} entri terakhir log audit"

def t_reorder():
    aid, _ = pick_audit()
    cur.execute("UPDATE CUSTOMER_AUDIT_LOG SET AUDIT_ID = -1 WHERE AUDIT_ID = :a", {"a": aid})
    cur.execute("UPDATE CUSTOMER_AUDIT_LOG SET AUDIT_ID = :a WHERE AUDIT_ID = :b", {"a": aid, "b": aid + 1})
    cur.execute("UPDATE CUSTOMER_AUDIT_LOG SET AUDIT_ID = :b WHERE AUDIT_ID = -1", {"b": aid + 1})
    return f"tukar urutan entri {aid} dan {aid+1}"

def t_history_laundering():
    # ubah langsung, lalu biarkan transaksi resmi berikutnya 'menimpa' baris tersebut
    cid = pick_live()
    cur.execute("UPDATE ABC_CUSTOMERS SET CONTRACT_POWER = CONTRACT_POWER + 2200 WHERE CUSTOMER_ID=:c", {"c": cid})
    cur.callproc("ABC_CDIVF.update_customer", [cid], {"p_address": "Jl. Pemutakhiran Resmi No. 9", "p_actor": "CS-PORTAL"})
    return f"ubah langsung pelanggan {cid}, lalu UPDATE resmi menimpanya"

def run_attack(name, fn):
    desc = fn()
    t0 = time.perf_counter(); info, viol = verify(); t_ms = (time.perf_counter() - t0) * 1000
    conn.rollback()
    return desc, info, viol, t_ms

SCENARIOS = [
    ("S2a", "Modifikasi CUSTOMER_NAME", t_modify("CUSTOMER_NAME"), {"RECORD_MODIFIED"}),
    ("S2b", "Modifikasi TARIFF_CODE", t_modify("TARIFF_CODE"), {"RECORD_MODIFIED"}),
    ("S2c", "Modifikasi CONTRACT_POWER", t_modify("CONTRACT_POWER"), {"RECORD_MODIFIED"}),
    ("S2d", "Modifikasi CUSTOMER_STATUS", t_modify("CUSTOMER_STATUS"), {"RECORD_MODIFIED"}),
    ("S3", "Penghapusan rekaman pelanggan", t_delete_record, {"RECORD_DELETED"}),
    ("S4", "Penyisipan tidak sah", t_unauth_insert, {"UNAUTHORIZED_INSERT"}),
    ("S5", "Modifikasi entri audit historis", t_audit_modify("ACTOR_ID"), {"ENTRY_HASH_MISMATCH"}),
    ("S6", "Manipulasi satu chain hash", t_chain_hash, {"ENTRY_HASH_MISMATCH", "CHAIN_LINK_BROKEN"}),
    ("S6b", "Penghapusan entri audit (tengah)", t_audit_delete, {"SEQUENCE_GAP", "CHAIN_LINK_BROKEN"}),
    ("S6c", "Pemotongan ekor log audit", t_tail_truncate, {"HEAD_MISMATCH"}),
    ("S6d", "Penukaran urutan entri", t_reorder, {"ENTRY_HASH_MISMATCH", "CHAIN_LINK_BROKEN"}),
    ("S6e", "History laundering", t_history_laundering, {"HISTORY_GAP"}),
]
attack_rows, attack_details = [], {}
for sid, title, fn, expected in SCENARIOS:
    desc, info, viol, t_ms = run_attack(sid, fn)
    types = set(viol.VIOLATION_TYPE)
    attack_rows.append({"skenario": sid, "serangan": title, "aksi": desc, "status": info.STATUS,
                        "jumlah_pelanggaran": int(info.VIOLATION_COUNT),
                        "jenis_terdeteksi": ", ".join(sorted(types)),
                        "klasifikasi_tepat": expected <= types, "waktu_verifikasi_ms": round(t_ms, 1)})
    attack_details[sid] = viol
attack_df = pd.DataFrame(attack_rows)
attack_df

,skenario,serangan,aksi,status,jumlah_pelanggaran,jenis_terdeteksi,klasifikasi_tepat,waktu_verifikasi_ms
0,S2a,Modifikasi CUSTOMER_NAME,UPDATE langsung CUSTOMER_NAME pelanggan 566,INVALID,1,RECORD_MODIFIED,True,23.8
1,S2b,Modifikasi TARIFF_CODE,UPDATE langsung TARIFF_CODE pelanggan 619,INVALID,1,RECORD_MODIFIED,True,19.4
2,S2c,Modifikasi CONTRACT_POWER,UPDATE langsung CONTRACT_POWER pelanggan 502,INVALID,1,RECORD_MODIFIED,True,23.0
3,S2d,Modifikasi CUSTOMER_STATUS,UPDATE langsung CUSTOMER_STATUS pelanggan 342,INVALID,1,RECORD_MODIFIED,True,23.5
4,S3,Penghapusan rekaman pelanggan,DELETE langsung pelanggan 72,INVALID,1,RECORD_DELETED,True,37.5
5,S4,Penyisipan tidak sah,INSERT langsung pelanggan tanpa entri audit,INVALID,1,UNAUTHORIZED_INSERT,True,19.7
6,S5,Modifikasi entri audit historis,UPDATE ACTOR_ID pada entri audit historis 685,INVALID,1,ENTRY_HASH_MISMATCH,True,21.2
7,S6,Manipulasi satu chain hash,ganti CURRENT_CHAIN_HASH entri 425,INVALID,2,"CHAIN_LINK_BROKEN, ENTRY_HASH_MISMATCH",True,21.1
8,S6b,Penghapusan entri audit (tengah),hapus entri audit 349 di tengah rantai,INVALID,3,"CHAIN_LINK_BROKEN, SEQUENCE_GAP, UNAUTHORIZED_INSERT",True,21.7
9,S6c,Pemotongan ekor log audit,potong 5 entri terakhir log audit,INVALID,6,"HEAD_MISMATCH, RECORD_MODIFIED",True,24.8


In [12]:
# Contoh rincian pelanggaran: lokalisasi manipulasi chain hash (S6)
attack_details["S6"]

,VIOLATION_TYPE,AUDIT_ID,CUSTOMER_ID,DETAIL
0,ENTRY_HASH_MISMATCH,425,425,hash dihitung ulang 72d8ef306edf7f11... <> tersimpan
1,CHAIN_LINK_BROKEN,426,426,PREVIOUS_CHAIN_HASH tidak sama dengan CURRENT_CHAIN_HASH entri sebelumnya


In [13]:
info, _ = verify(); conn.commit()
print("Setelah semua rollback, status rantai:", info.STATUS)
assert info.STATUS == "VALID"

Setelah semua rollback, status rantai: VALID


## Fase 7 — Skenario 9 (tambahan): penulisan ulang seluruh rantai oleh penyerang berprivilese penuh

Hash chaining **tidak mencegah** perusakan. Mekanisme ini hanya menyediakan **bukti perusakan** (*tamper evidence*). Penyerang yang menguasai seluruh basis data dapat mengubah sebuah rekaman, menghitung ulang `AFTER_DATA_HASH` dan semua chain hash berikutnya, lalu memperbarui `CHAIN_HEAD` dan `INTEGRITY_CHECKPOINT` internal. Skenario ini menguji batas tersebut sekaligus nilai **jangkar eksternal**.

In [14]:
# pilih pelanggan yang entri terakhirnya berada SEBELUM checkpoint (agar checkpoint ikut ditulis ulang)
cid_rw = int(db.query_df(conn, """SELECT CUSTOMER_ID FROM (SELECT CUSTOMER_ID, MAX(AUDIT_ID) M FROM CUSTOMER_AUDIT_LOG
    GROUP BY CUSTOMER_ID HAVING MAX(AUDIT_ID) < :cp
    AND SUM(CASE WHEN OPERATION_TYPE = 'DELETE' THEN 1 ELSE 0 END) = 0 ORDER BY M) WHERE ROWNUM = 1""", {"cp": int(cp1.LAST_AUDIT_ID)}).iloc[0, 0])
REWRITE = """
DECLARE
  k NUMBER; prev VARCHAR2(64); newh VARCHAR2(64); newa VARCHAR2(64); r ABC_CUSTOMERS%ROWTYPE;
BEGIN
  UPDATE ABC_CUSTOMERS SET TARIFF_CODE = 'R-1/TR', CONTRACT_POWER = 450 WHERE CUSTOMER_ID = :cid;
  SELECT * INTO r FROM ABC_CUSTOMERS WHERE CUSTOMER_ID = :cid;
  SELECT MAX(AUDIT_ID) INTO k FROM CUSTOMER_AUDIT_LOG WHERE CUSTOMER_ID = :cid;
  newa := ABC_CDIVF.record_hash(r);
  UPDATE CUSTOMER_AUDIT_LOG SET AFTER_DATA_HASH = newa WHERE AUDIT_ID = k;
  SELECT PREVIOUS_CHAIN_HASH INTO prev FROM CUSTOMER_AUDIT_LOG WHERE AUDIT_ID = k;
  FOR a IN (SELECT * FROM CUSTOMER_AUDIT_LOG WHERE AUDIT_ID >= k ORDER BY AUDIT_ID) LOOP
    newh := ABC_CDIVF.chain_hash(prev, ABC_CDIVF.canon_tx(a.AUDIT_ID, a.TRANSACTION_ID, a.CUSTOMER_ID,
              a.OPERATION_TYPE, a.TRANSACTION_TIMESTAMP, a.BEFORE_DATA_HASH, a.AFTER_DATA_HASH, a.ACTOR_ID));
    UPDATE CUSTOMER_AUDIT_LOG SET PREVIOUS_CHAIN_HASH = prev, CURRENT_CHAIN_HASH = newh WHERE AUDIT_ID = a.AUDIT_ID;
    UPDATE INTEGRITY_CHECKPOINT SET LAST_CHAIN_HASH = newh WHERE LAST_AUDIT_ID = a.AUDIT_ID;
    prev := newh;
  END LOOP;
  UPDATE CHAIN_HEAD SET LAST_CHAIN_HASH = prev WHERE CHAIN_ID = 1;
END;"""
t0 = time.perf_counter(); cur.execute(REWRITE, {"cid": cid_rw}); rw_ms = (time.perf_counter() - t0) * 1000
n_rewritten = int(db.query_df(conn, "SELECT COUNT(*) FROM CUSTOMER_AUDIT_LOG WHERE AUDIT_ID >= "
    "(SELECT MAX(AUDIT_ID) FROM CUSTOMER_AUDIT_LOG WHERE CUSTOMER_ID=:c)", {"c": cid_rw}).iloc[0, 0])
info_int, viol_int = verify()
ext = anchors.latest()
py_ext = cn.independent_verify(conn, trusted_anchor=ext)
conn.rollback()
s9 = pd.DataFrame([
    ["Verifikasi internal (log + checkpoint internal)", info_int.STATUS, int(info_int.VIOLATION_COUNT)],
    ["Verifikasi dengan jangkar eksternal", "INVALID" if py_ext else "VALID", len(py_ext)],
], columns=["mekanisme", "status", "pelanggaran"])
print(f"Pelanggan {cid_rw}: {n_rewritten} entri rantai dihitung ulang oleh penyerang dalam {rw_ms:.0f} ms")
print("Pelanggaran via jangkar eksternal:", py_ext)
s9

Pelanggan 4: 1427 entri rantai dihitung ulang oleh penyerang dalam 37 ms
Pelanggaran via jangkar eksternal: [('EXTERNAL_ANCHOR_MISMATCH', 1330, None)]


,mekanisme,status,pelanggaran
0,Verifikasi internal (log + checkpoint internal),VALID,0
1,Verifikasi dengan jangkar eksternal,INVALID,1


## Fase 8 — Tingkat deteksi (TDR) dan *false positive rate* (FPR)

Setiap jenis serangan diulang `TRIALS_PER_TYPE` kali dengan target acak. Kolom yang dimodifikasi pada entri audit juga diacak. Verifikasi menghasilkan *true positive* jika status INVALID. FPR dihitung dari verifikasi atas keadaan bersih setelah beban kerja sah acak, termasuk beban kerja konkuren.

In [15]:
TRIAL_TYPES = [
    ("Modifikasi rekaman", lambda: t_modify(random.choice(["CUSTOMER_NAME", "TARIFF_CODE", "CONTRACT_POWER", "CUSTOMER_STATUS"]))()),
    ("Penghapusan rekaman", t_delete_record),
    ("Penyisipan tidak sah", t_unauth_insert),
    ("Modifikasi entri audit", t_audit_modify(None)),
    ("Manipulasi chain hash", t_chain_hash),
    ("Penghapusan entri audit", t_audit_delete),
    ("Pemotongan ekor log", t_tail_truncate),
    ("Penukaran urutan entri", t_reorder),
    ("History laundering", t_history_laundering),
]
trial_rows = []
for name, fn in TRIAL_TYPES:
    for t in range(TRIALS_PER_TYPE):
        fn(); info, viol = verify(); py = cn.independent_verify(conn); conn.rollback()
        trial_rows.append({"serangan": name, "trial": t, "plsql_detect": info.STATUS == "INVALID",
                           "python_detect": bool(py)})
# Uji negatif: keadaan bersih setelah beban kerja sah acak
for t in range(TRIALS_PER_TYPE):
    for _ in range(random.randint(5, 30)):
        op = random.random()
        if op < 0.5:
            x = gen.customer()
            live_ids.append(acur.callfunc("ABCSEC.ABC_CDIVF.insert_customer", int,
                [x["name"], x["number"], x["address"], x["tariff"], x["power"], x["meter"], "AKTIF", "CS-PORTAL"]))
        elif op < 0.9:
            acur.callproc("ABCSEC.ABC_CDIVF.update_customer", [random.choice(live_ids)], gen.change())
        else:
            victim = live_ids.pop(random.randrange(len(live_ids)))
            acur.callproc("ABCSEC.ABC_CDIVF.delete_customer", [victim, "CS-PORTAL"])
        app.commit()
    if t % 10 == 0:
        concurrent_workload(4, 10, seed=100 + t)
    info, _ = verify(); py = cn.independent_verify(conn); conn.commit()
    trial_rows.append({"serangan": "(bersih — uji negatif)", "trial": t, "plsql_detect": info.STATUS == "INVALID",
                       "python_detect": bool(py)})
trials = pd.DataFrame(trial_rows)
tdr = trials.groupby("serangan", sort=False).agg(trial=("trial", "count"),
        terdeteksi_plsql=("plsql_detect", "sum"), terdeteksi_python=("python_detect", "sum"))
tdr["rate_plsql_%"] = 100 * tdr.terdeteksi_plsql / tdr.trial
tdr["rate_python_%"] = 100 * tdr.terdeteksi_python / tdr.trial
attacks = trials[trials.serangan != "(bersih — uji negatif)"]
clean = trials[trials.serangan == "(bersih — uji negatif)"]
TDR = 100 * attacks.plsql_detect.mean(); FPR = 100 * clean.plsql_detect.mean()
print(f"Tampering Detection Rate (PL/SQL): {TDR:.1f}%  dari {len(attacks)} serangan")
print(f"False Positive Rate           : {FPR:.1f}%  dari {len(clean)} verifikasi bersih")
tdr

Tampering Detection Rate (PL/SQL): 100.0%  dari 270 serangan
False Positive Rate           : 0.0%  dari 30 verifikasi bersih


,trial,terdeteksi_plsql,terdeteksi_python,rate_plsql_%,rate_python_%
serangan,,,,,
Modifikasi rekaman,30,30,30,100.0,100.0
Penghapusan rekaman,30,30,30,100.0,100.0
Penyisipan tidak sah,30,30,30,100.0,100.0
Modifikasi entri audit,30,30,30,100.0,100.0
Manipulasi chain hash,30,30,30,100.0,100.0
Penghapusan entri audit,30,30,30,100.0,100.0
Pemotongan ekor log,30,30,30,100.0,100.0
Penukaran urutan entri,30,30,30,100.0,100.0
History laundering,30,30,30,100.0,100.0


## Fase 9 — Skenario 8: pemulihan dari checkpoint terpercaya dan verifikasi ulang

1. Buat checkpoint C2 dan jangkarkan ke penyimpanan eksternal.
2. Jalankan transaksi sah setelah C2 dan catat di jurnal aplikasi.
3. Penyerang **meng-commit** perusakan rekaman dan entri audit.
4. Verifikasi mendeteksi pelanggaran. Verifikasi inkremental dari C2 melokalisasi segmen rusak.
5. Pulihkan tabel ke SCN checkpoint C2 dengan Oracle Flashback Query. Validasi C2 terhadap jangkar eksternal.
6. Putar ulang jurnal transaksi sah melalui paket, lalu verifikasi ulang.

In [16]:
cp2 = checkpoint_and_anchor()
assert cp2.VERIFICATION_STATUS == "VALID"
journal = []
for _ in range(50):
    cid = random.choice(live_ids); ch = gen.change()
    acur.callproc("ABCSEC.ABC_CDIVF.update_customer", [cid], dict(ch, p_actor="CS-PORTAL")); app.commit()
    journal.append((cid, ch))
# serangan yang di-commit
victim = pick_live()
cur.execute("UPDATE ABC_CUSTOMERS SET TARIFF_CODE='B-2', CONTRACT_POWER=11000 WHERE CUSTOMER_ID=:c", {"c": victim})
aid_t = int(cp2.LAST_AUDIT_ID) + 10   # entri setelah checkpoint C2
cur.execute("UPDATE CUSTOMER_AUDIT_LOG SET ACTOR_ID='SYSTEM@ABC_APP' WHERE AUDIT_ID=:a", {"a": aid_t})
conn.commit()

info_full, viol_full = verify(); conn.commit()
info_inc, viol_inc = verify(int(cp2.CHECKPOINT_ID)); conn.commit()
print(f"Verifikasi penuh      : {info_full.STATUS}, {info_full.VIOLATION_COUNT} pelanggaran, {info_full.ELAPSED_MS:.1f} ms")
print(f"Verifikasi dari C2    : {info_inc.STATUS}, {info_inc.VIOLATION_COUNT} pelanggaran, {info_inc.ELAPSED_MS:.1f} ms "
      f"(entri diperiksa {info_inc.ENTRIES_CHECKED})")
display(viol_full)

# Pemulihan: Flashback Query ke SCN checkpoint (keadaan terverifikasi)
scn = int(cp2.CHECKPOINT_SCN)
t0 = time.perf_counter()
for tbl in ["CUSTOMER_AUDIT_LOG", "ABC_CUSTOMERS"]:
    cur.execute(f"DELETE FROM {tbl}")
    cur.execute(f"INSERT INTO {tbl} SELECT * FROM {tbl} AS OF SCN :s", {"s": scn})
cur.execute("""UPDATE CHAIN_HEAD h SET (LAST_AUDIT_ID, LAST_CHAIN_HASH, UPDATED_AT) =
               (SELECT LAST_AUDIT_ID, LAST_CHAIN_HASH, UPDATED_AT FROM CHAIN_HEAD AS OF SCN :s WHERE CHAIN_ID=1)""", {"s": scn})
conn.commit()
restore_ms = (time.perf_counter() - t0) * 1000
ext = [e for e in anchors.entries() if e["checkpoint_id"] == int(cp2.CHECKPOINT_ID)][0]
head = db.query_df(conn, "SELECT LAST_AUDIT_ID, LAST_CHAIN_HASH FROM CHAIN_HEAD").iloc[0]
print(f"Pemulihan selesai {restore_ms:.0f} ms; kepala rantai = jangkar eksternal C2:",
      head.LAST_CHAIN_HASH == ext["LAST_CHAIN_HASH"] and head.LAST_AUDIT_ID == ext["LAST_AUDIT_ID"])
info_rest, _ = verify(); conn.commit()
print("Verifikasi setelah pemulihan:", info_rest.STATUS)

# Putar ulang transaksi sah dari jurnal aplikasi
for cid, ch in journal:
    acur.callproc("ABCSEC.ABC_CDIVF.update_customer", [cid], dict(ch, p_actor="REPLAY")); app.commit()
info_replay, _ = verify(); conn.commit()
py = cn.independent_verify(conn, trusted_anchor=anchors.latest())
print(f"Setelah replay {len(journal)} transaksi: {info_replay.STATUS}; verifikator Python + jangkar: "
      f"{'VALID' if not py else py}")
recovery = pd.DataFrame([
    ["Setelah serangan (penuh)", info_full.STATUS, info_full.VIOLATION_COUNT, info_full.ENTRIES_CHECKED, info_full.ELAPSED_MS],
    ["Setelah serangan (inkremental dari C2)", info_inc.STATUS, info_inc.VIOLATION_COUNT, info_inc.ENTRIES_CHECKED, info_inc.ELAPSED_MS],
    ["Setelah flashback ke C2", info_rest.STATUS, info_rest.VIOLATION_COUNT, info_rest.ENTRIES_CHECKED, info_rest.ELAPSED_MS],
    ["Setelah replay jurnal", info_replay.STATUS, info_replay.VIOLATION_COUNT, info_replay.ENTRIES_CHECKED, info_replay.ELAPSED_MS],
], columns=["tahap", "status", "pelanggaran", "entri_diperiksa", "waktu_ms"])
assert info_rest.STATUS == "VALID" and info_replay.STATUS == "VALID" and not py
recovery

Verifikasi penuh      : INVALID, 2 pelanggaran, 25.5 ms
Verifikasi dari C2    : INVALID, 2 pelanggaran, 12.1 ms (entri diperiksa 50)


,VIOLATION_TYPE,AUDIT_ID,CUSTOMER_ID,DETAIL
0,ENTRY_HASH_MISMATCH,2128,259,hash dihitung ulang 2f4f1174f4565140... <> tersimpan
1,RECORD_MODIFIED,1463,838,hash rekaman saat ini tidak sama dengan AFTER_DATA_HASH terakhir


Pemulihan selesai 43 ms; kepala rantai = jangkar eksternal C2: True
Verifikasi setelah pemulihan: VALID


Setelah replay 50 transaksi: VALID; verifikator Python + jangkar: VALID


,tahap,status,pelanggaran,entri_diperiksa,waktu_ms
0,Setelah serangan (penuh),INVALID,2,2168,25.458
1,Setelah serangan (inkremental dari C2),INVALID,2,50,12.103
2,Setelah flashback ke C2,VALID,0,2118,26.058
3,Setelah replay jurnal,VALID,0,2168,24.867


In [17]:
# simpan hasil skenario fungsional ke EXPERIMENT_RESULTS
n_chain = int(db.query_df(conn, "SELECT COUNT(*) FROM CUSTOMER_AUDIT_LOG").iloc[0, 0])
for r in attack_rows:
    cur.execute("""INSERT INTO EXPERIMENT_RESULTS (EXPERIMENT_SCENARIO, RECORD_COUNT, VERIFICATION_TIME_MS,
        DETECTED_TAMPERING_COUNT, MODE_NAME, NOTES) VALUES (:s, :n, :v, :d, 'CHAINED', :x)""",
        {"s": r["skenario"], "n": n_chain, "v": r["waktu_verifikasi_ms"], "d": r["jumlah_pelanggaran"],
         "x": (r["serangan"] + " | " + r["jenis_terdeteksi"])[:400]})
cur.execute("""INSERT INTO EXPERIMENT_RESULTS (EXPERIMENT_SCENARIO, RECORD_COUNT, VERIFICATION_TIME_MS,
    DETECTED_TAMPERING_COUNT, MODE_NAME, NOTES) VALUES ('S8_RECOVERY', :n, :v, :d, 'CHAINED', :x)""",
    {"n": n_chain, "v": float(info_replay.ELAPSED_MS), "d": int(info_full.VIOLATION_COUNT),
     "x": "deteksi -> flashback ke checkpoint -> replay -> VALID"})
conn.commit()
attack_df.to_csv(RESULTS / "tamper_scenarios.csv", index=False)
tdr.to_csv(RESULTS / "detection_rate.csv")
trials.to_csv(RESULTS / "detection_trials.csv", index=False)
s9.to_csv(RESULTS / "full_rewrite.csv", index=False)
recovery.to_csv(RESULTS / "recovery.csv", index=False)
canon_df.to_csv(RESULTS / "canonical_vectors.csv", index=False)
func_state = {"n_chain": n_chain, "TDR": TDR, "FPR": FPR, "n_attacks": len(attacks), "n_clean": len(clean)}
json.dump(func_state, open(RESULTS / "functional_summary.json", "w"), indent=2)
func_state

{'n_chain': 2168,
 'TDR': np.float64(100.0),
 'FPR': np.float64(0.0),
 'n_attacks': 270,
 'n_clean': 30}

## Fase 10 — Skenario 7: evaluasi kinerja skala besar

Prosedur `ABC_EXPERIMENT.run_scaling(n)` dijalankan **di dalam basis data**. Untuk setiap repetisi, prosedur ini:

1. mengosongkan tabel;
2. **baseline**: INSERT *n*, UPDATE *n*/10, DELETE *n*/100 pada tabel tanpa audit dan tanpa kriptografi;
3. **audit konvensional**: operasi yang sama pada tabel ber-*trigger* audit (tanpa hash), untuk memisahkan biaya pencatatan audit dari biaya kriptografi;
4. **ABC-CDIVF**: operasi yang sama melalui paket hash chaining;
5. mengukur biaya murni pembangkitan hash untuk *n* transaksi (serialisasi + 2×SHA-256, tanpa DML);
6. menjalankan verifikasi penuh (rantai + keadaan).

*Commit* dilakukan setiap 1.000 operasi pada kedua mode. Waktu diukur dengan `SYSTIMESTAMP`, CPU dengan `DBMS_UTILITY.GET_CPU_TIME`, dan memori dengan statistik `session pga memory max`.

In [18]:
cur.execute("DELETE FROM EXPERIMENT_RESULTS WHERE EXPERIMENT_SCENARIO LIKE 'S7%'"); conn.commit()
storage = []
SEG_SQL = """SELECT grp, SUM(b) FROM (SELECT CASE WHEN segment_name IN ('ABC_CUSTOMERS','PK_ABC_CUSTOMERS','UQ_ABC_CUSTOMER_NUMBER') THEN 'data_pelanggan'
                    WHEN segment_name IN ('ABC_CUSTOMERS_BASELINE','PK_ABC_CUSTOMERS_BASE','UQ_ABC_CUSTOMER_NUMBER_BASE') THEN 'baseline'
                    WHEN segment_name IN ('CUSTOMER_AUDIT_LOG','PK_CUSTOMER_AUDIT_LOG','IX_AUDIT_CUSTOMER') THEN 'log_audit_rantai'
                    WHEN segment_name IN ('CUSTOMER_AUDIT_PLAIN','PK_CUSTOMER_AUDIT_PLAIN') THEN 'log_audit_konvensional'
                    ELSE 'lain' END grp, bytes b FROM user_segments) GROUP BY grp"""
wall = {}
for n in SIZES:
    t0 = time.perf_counter()
    for rep in range(1, REPS[n] + 1):
        cur.callproc("ABC_EXPERIMENT.run_scaling", [n, rep])
    wall[n] = time.perf_counter() - t0
    seg = dict(db.query_df(conn, SEG_SQL).values)
    exact = db.query_df(conn, """SELECT (SELECT SUM(VSIZE(AUDIT_ID)+VSIZE(TRANSACTION_ID)+VSIZE(CUSTOMER_ID)+VSIZE(OPERATION_TYPE)
            +VSIZE(TRANSACTION_TIMESTAMP)+NVL(VSIZE(BEFORE_DATA_HASH),0)+NVL(VSIZE(AFTER_DATA_HASH),0)
            +VSIZE(PREVIOUS_CHAIN_HASH)+VSIZE(CURRENT_CHAIN_HASH)+VSIZE(ACTOR_ID)) FROM CUSTOMER_AUDIT_LOG) AUDIT_B,
            (SELECT COUNT(*) FROM CUSTOMER_AUDIT_LOG) AUDIT_N, (SELECT COUNT(*) FROM ABC_CUSTOMERS) CUST_N FROM dual""").iloc[0]
    storage.append({"n": n, "segmen_pelanggan_MB": seg.get("data_pelanggan", 0) / 2**20,
                    "segmen_log_audit_MB": seg.get("log_audit_rantai", 0) / 2**20,
                    "segmen_log_konvensional_MB": seg.get("log_audit_konvensional", 0) / 2**20,
                    "entri_audit": int(exact.AUDIT_N), "rekaman_aktif": int(exact.CUST_N),
                    "byte_data_per_entri_audit": float(exact.AUDIT_B) / int(exact.AUDIT_N)})
    print(f"n={n:>9,}: {REPS[n]} repetisi selesai dalam {wall[n]:.1f} s")
storage_df = pd.DataFrame(storage)
storage_df["overhead_penyimpanan_%"] = 100 * storage_df.segmen_log_audit_MB / storage_df.segmen_pelanggan_MB
raw = db.query_df(conn, "SELECT * FROM EXPERIMENT_RESULTS WHERE EXPERIMENT_SCENARIO LIKE 'S7%' ORDER BY EXPERIMENT_ID")
raw.to_csv(RESULTS / "scaling_raw.csv", index=False)
storage_df.to_csv(RESULTS / "storage_overhead.csv", index=False)
storage_df

n=    1,000: 5 repetisi selesai dalam 1.2 s


n=   10,000: 5 repetisi selesai dalam 10.5 s


n=  100,000: 5 repetisi selesai dalam 101.2 s


n=1,000,000: 3 repetisi selesai dalam 701.5 s


,n,segmen_pelanggan_MB,segmen_log_audit_MB,segmen_log_konvensional_MB,entri_audit,rekaman_aktif,byte_data_per_entri_audit,overhead_penyimpanan_%
0,1000,0.3125,0.5625,0.1875,1110,990,240.476577,180.000000
1,10000,2.6250,4.8125,1.0000,11100,9900,240.817207,183.333333
2,100000,19.0000,40.0000,9.0000,111000,99000,242.265333,210.526316
3,1000000,176.0000,379.0000,81.0000,1110000,990000,243.284009,215.340909


In [19]:
def ms(s): return s.astype(float)
tot = raw[raw.EXPERIMENT_SCENARIO == "S7_CRUD_TOTAL"].copy()
ins = raw[raw.EXPERIMENT_SCENARIO == "S7_INSERT"].copy()
ver = raw[raw.EXPERIMENT_SCENARIO == "S7_VERIFY"].copy()
rows = []
for n in SIZES:
    b = ms(tot[(tot.RECORD_COUNT == n) & (tot.MODE_NAME == "BASELINE")].TOTAL_PROCESSING_TIME_MS)
    h = ms(tot[(tot.RECORD_COUNT == n) & (tot.MODE_NAME == "CHAINED")].TOTAL_PROCESSING_TIME_MS)
    bi = ms(ins[(ins.RECORD_COUNT == n) & (ins.MODE_NAME == "BASELINE")].TOTAL_PROCESSING_TIME_MS)
    hi = ms(ins[(ins.RECORD_COUNT == n) & (ins.MODE_NAME == "CHAINED")].TOTAL_PROCESSING_TIME_MS)
    a = ms(tot[(tot.RECORD_COUNT == n) & (tot.MODE_NAME == "AUDIT_ONLY")].TOTAL_PROCESSING_TIME_MS)
    hg = ms(tot[(tot.RECORD_COUNT == n) & (tot.MODE_NAME == "CHAINED")].HASH_GENERATION_TIME_MS)
    v = ver[ver.REPETITION.isin(tot[tot.RECORD_COUNT == n].REPETITION) & (ver.RECORD_COUNT == n + n // 10 + n // 100)]
    vt = ms(v.VERIFICATION_TIME_MS)
    ops = n + n // 10 + n // 100
    cpu_b = ms(tot[(tot.RECORD_COUNT == n) & (tot.MODE_NAME == "BASELINE")].CPU_TIME_MS)
    cpu_h = ms(tot[(tot.RECORD_COUNT == n) & (tot.MODE_NAME == "CHAINED")].CPU_TIME_MS)
    t_stat, p_val = stats.ttest_ind(h, b, equal_var=False)
    rows.append({
        "n": n, "repetisi": len(b),
        "CRUD_baseline_ms": b.mean(), "sd_baseline": b.std(ddof=1),
        "CRUD_audit_ms": a.mean(), "sd_audit": a.std(ddof=1),
        "CRUD_chained_ms": h.mean(), "sd_chained": h.std(ddof=1),
        "overhead_%": 100 * (h.mean() - b.mean()) / b.mean(),
        "overhead_vs_audit_%": 100 * (h.mean() - a.mean()) / a.mean(),
        "INSERT_baseline_ms": bi.mean(), "INSERT_chained_ms": hi.mean(),
        "us_per_op_baseline": 1000 * b.mean() / ops, "us_per_op_audit": 1000 * a.mean() / ops,
        "us_per_op_chained": 1000 * h.mean() / ops,
        "hash_gen_ms": hg.mean(), "sd_hash_gen": hg.std(ddof=1), "us_hash_per_tx": 1000 * hg.mean() / n,
        "verifikasi_ms": vt.mean(), "sd_verifikasi": vt.std(ddof=1),
        "us_verif_per_entri": 1000 * vt.mean() / ops,
        "throughput_baseline_ops": ops / (b.mean() / 1000), "throughput_chained_ops": ops / (h.mean() / 1000),
        "throughput_verif_entri_s": ops / (vt.mean() / 1000),
        "CPU_baseline_ms": cpu_b.mean(), "CPU_chained_ms": cpu_h.mean(),
        "PGA_max_MB": ms(v.PGA_MAX_BYTES).max() / 2**20,
        "welch_t": t_stat, "p_value": p_val,
    })
perf = pd.DataFrame(rows)
perf.to_csv(RESULTS / "scaling_summary.csv", index=False)
pd.options.display.float_format = "{:,.3f}".format
perf.T

,0,1,2,3
n,"1,000.000","10,000.000","100,000.000","1,000,000.000"
repetisi,5.000,5.000,5.000,3.000
CRUD_baseline_ms,12.702,184.144,"2,218.426","26,716.333"
sd_baseline,1.449,43.221,60.982,"1,336.229"
CRUD_audit_ms,50.512,435.264,"4,643.743","57,561.999"
sd_audit,51.993,37.732,244.981,"1,882.275"
CRUD_chained_ms,93.805,"1,087.161","9,926.113","113,019.932"
sd_chained,48.733,227.236,470.305,"1,616.123"
overhead_%,638.531,490.385,347.439,323.037
overhead_vs_audit_%,85.710,149.770,113.752,96.345


In [20]:
# Gambar 1: waktu CRUD baseline vs ABC-CDIVF (log-log) dengan error bar (±1 SD)
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8))
ax = axes[0]
ax.errorbar(perf.n, perf.CRUD_baseline_ms, yerr=perf.sd_baseline, marker="o", ms=4, lw=2, color=C_BASE,
            capsize=3, label="CRUD konvensional")
ax.errorbar(perf.n, perf.CRUD_chained_ms, yerr=perf.sd_chained, marker="s", ms=4, lw=2, color=C_CHAIN,
            capsize=3, label="CRUD + SHA-256 hash chaining")
ax.errorbar(perf.n, perf.CRUD_audit_ms, yerr=perf.sd_audit, marker="D", ms=4, lw=2, color=C_AUDIT,
            capsize=3, label="CRUD + audit trigger")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Jumlah rekaman (n)"); ax.set_ylabel("Waktu total CRUD (ms)")
ax.set_title("(a) Waktu pemrosesan transaksi", color=C_INK)
ax.legend(frameon=False, fontsize=7.5)
ax = axes[1]
ax.errorbar(perf.n, perf.verifikasi_ms, yerr=perf.sd_verifikasi, marker="o", ms=4, lw=2, color=C_CHAIN,
            capsize=3, label="verifikasi penuh")
ax.errorbar(perf.n, perf.hash_gen_ms, yerr=perf.sd_hash_gen, marker="^", ms=4, lw=2, color=C_BASE,
            capsize=3, label="pembangkitan hash murni")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Jumlah rekaman (n)"); ax.set_ylabel("Waktu (ms)")
ax.set_title("(b) Verifikasi dan pembangkitan hash", color=C_INK)
ax.legend(frameon=False, fontsize=7.5)
fig.tight_layout(); fig.savefig(FIGS / "fig_scaling.png", dpi=300); plt.show()

In [21]:
# Gambar 2: biaya per operasi (mikrodetik) — memperlihatkan perilaku linear O(n)
fig, ax = plt.subplots(figsize=(3.5, 2.6))
x = np.arange(len(perf)); w = 0.27
ax.bar(x - w - 0.02, perf.us_per_op_baseline, w, color=C_BASE, label="CRUD konvensional")
ax.bar(x, perf.us_per_op_audit, w, color=C_AUDIT, label="+ audit trigger")
ax.bar(x + w + 0.02, perf.us_per_op_chained, w, color=C_CHAIN, label="+ hash chaining (ABC-CDIVF)")
for i, r in perf.iterrows():
    ax.text(i + w + 0.02, r.us_per_op_chained, f"+{r['overhead_%']:.0f}%", ha="center", va="bottom", fontsize=7, color=C_INK)
ax.set_xticks(x, [f"{n:,}".replace(",", ".") for n in perf.n])
ax.set_xlabel("Jumlah rekaman (n)"); ax.set_ylabel("µs per operasi")
ax.legend(frameon=False, fontsize=6.5, loc="upper left"); ax.set_ylim(0, perf.us_per_op_chained.max() * 1.45)
fig.tight_layout(); fig.savefig(FIGS / "fig_per_op.png", dpi=300); plt.show()

## Fase 11 — Verifikasi inkremental dan skalabilitas konkurensi

In [22]:
# Verifikasi inkremental: checkpoint pada n=100.000 entri lalu 1.000 transaksi tambahan
n_inc = 10_000 if QUICK else 100_000
cur.callproc("ABC_EXPERIMENT.run_scaling", [n_inc, 99])
cur.execute("DELETE FROM EXPERIMENT_RESULTS WHERE REPETITION = 99"); conn.commit()
cp_inc = cur.callfunc("ABC_CDIVF.create_checkpoint", int, [0]); conn.commit()
cur.execute("""BEGIN FOR c IN (SELECT CUSTOMER_ID FROM ABC_CUSTOMERS WHERE ROWNUM <= 1000) LOOP
  ABC_CDIVF.update_customer(c.CUSTOMER_ID, p_status => 'NONAKTIF', p_actor => 'INC'); END LOOP; COMMIT; END;""")
inc_rows = []
for rep in range(5):
    full_chain, _ = verify(None, 0); part_chain, _ = verify(cp_inc, 0); conn.rollback()
    inc_rows.append({"rep": rep, "penuh_ms": full_chain.CHAIN_ELAPSED_MS, "penuh_entri": full_chain.ENTRIES_CHECKED,
                     "inkremental_ms": part_chain.CHAIN_ELAPSED_MS, "inkremental_entri": part_chain.ENTRIES_CHECKED})
inc = pd.DataFrame(inc_rows)
inc_summary = inc.agg(["mean", "std"])
inc_summary["speedup"] = inc_summary.penuh_ms / inc_summary.inkremental_ms
inc.to_csv(RESULTS / "incremental_verification.csv", index=False)
inc_summary

,rep,penuh_ms,penuh_entri,inkremental_ms,inkremental_entri,speedup
mean,2.000,771.370,"112,000.000",7.523,"1,000.000",102.538
std,1.581,35.852,0.000,0.885,0.000,40.496


In [23]:
# Skalabilitas konkurensi: kepala rantai tunggal menserialkan penulis
cur.callproc("ABC_CDIVF.reset_all")
conc_rows = []
TOTAL_OPS = 600 if QUICK else 1200
for th in [1, 2, 4, 8]:
    for rep in range(3):
        el, errs = concurrent_workload(th, TOTAL_OPS // th, seed=1000 + th * 10 + rep)
        assert not errs, errs
        conc_rows.append({"sesi": th, "rep": rep, "detik": el, "ops_per_detik": TOTAL_OPS / el})
info, _ = verify(); conn.commit()
conc = pd.DataFrame(conc_rows).groupby("sesi").agg(ops_per_detik=("ops_per_detik", "mean"),
                                                    sd=("ops_per_detik", "std")).reset_index()
print("Status rantai setelah", int(info.ENTRIES_CHECKED), "operasi konkuren:", info.STATUS)
assert info.STATUS == "VALID"
conc.to_csv(RESULTS / "concurrency.csv", index=False)
conc

Status rantai setelah 14400 operasi konkuren: VALID


,sesi,ops_per_detik,sd
0,1,"3,214.537",820.728
1,2,"4,396.931","1,223.296"
2,4,"4,756.563","1,160.561"
3,8,"3,455.807","1,032.732"


In [24]:
fig, ax = plt.subplots(figsize=(3.5, 2.4))
ax.errorbar(conc.sesi, conc.ops_per_detik, yerr=conc.sd, marker="o", ms=4, lw=2, color=C_CHAIN, capsize=3)
ax.set_xticks(conc.sesi); ax.set_xlabel("Jumlah sesi penulis paralel"); ax.set_ylabel("Transaksi per detik")
ax.set_ylim(0, conc.ops_per_detik.max() * 1.25)
fig.tight_layout(); fig.savefig(FIGS / "fig_concurrency.png", dpi=300); plt.show()

## Fase 12 — Dataset demo untuk dashboard, ekspor hasil, dan ringkasan

In [25]:
# Dataset demo yang bersih untuk dashboard Streamlit (+ checkpoint dan jangkar eksternal)
cur.callproc("ABC_CDIVF.reset_all")
if anchor_path.exists(): anchor_path.unlink()
anchors = ExternalAnchorStore(anchor_path)
g2 = SyntheticCustomerGenerator(seed=7)
demo_ids = []
for i in range(2000):
    x = g2.customer()
    demo_ids.append(acur.callfunc("ABCSEC.ABC_CDIVF.insert_customer", int,
        [x["name"], x["number"], x["address"], x["tariff"], x["power"], x["meter"], "AKTIF", "CS-PORTAL"]))
    if i % 100 == 99: app.commit()
for i in range(600):
    acur.callproc("ABCSEC.ABC_CDIVF.update_customer", [random.choice(demo_ids)], dict(g2.change(), p_actor="CS-PORTAL"))
app.commit()
checkpoint_and_anchor()
info, _ = verify(); conn.commit()
print("Dataset demo:", info.ENTRIES_CHECKED, "entri rantai,", info.RECORDS_CHECKED, "pelanggan, status", info.STATUS)
env = {"oracle": db.query_df(conn, "SELECT banner_full FROM v$version").iloc[0, 0].split("\n")[0],
       "python": platform.python_version(), "machine": platform.machine(), "cpus_visible": os.cpu_count(),
       "quick_mode": QUICK, "sizes": SIZES, "reps": {str(k): v for k, v in REPS.items() if k in SIZES},
       "wall_seconds": {str(k): round(v, 1) for k, v in wall.items()}}
json.dump(env, open(RESULTS / "environment.json", "w"), indent=2)
env

Dataset demo: 2600 entri rantai, 2000 pelanggan, status VALID


{'oracle': 'Oracle AI Database 26ai Free Release 23.26.3.0.0 - Develop, Learn, and Run for Free',
 'python': '3.12.16',
 'machine': 'aarch64',
 'cpus_visible': 10,
 'quick_mode': False,
 'sizes': [1000, 10000, 100000, 1000000],
 'reps': {'1000': 5, '10000': 5, '100000': 5, '1000000': 3},
 'wall_seconds': {'1000': 1.2,
  '10000': 10.5,
  '100000': 101.2,
  '1000000': 701.5}}

### Ringkasan jawaban pertanyaan penelitian

Nilai-nilai pada sel berikut dihitung langsung dari hasil eksekusi notebook ini, bukan dari estimasi teoretis.

In [26]:
big = perf.iloc[-1]
print(f"RQ1  Implementasi: paket PL/SQL ABC_CDIVF (DBMS_CRYPTO SHA-256), serialisasi kanonik tervalidasi silang dengan Python.")
print(f"RQ2  TDR = {TDR:.1f}% ({len(attacks)} serangan, {len(TRIAL_TYPES)} jenis), FPR = {FPR:.1f}% ({len(clean)} uji bersih); "
      f"penulisan ulang rantai penuh hanya terdeteksi oleh jangkar eksternal.")
for _, r in perf.iterrows():
    print(f"RQ3  n={int(r.n):>9,}: overhead CRUD {r['overhead_%']:.1f}% "
          f"({r.us_per_op_baseline:.1f} -> {r.us_per_op_chained:.1f} µs/operasi), vs audit trigger "
          f"{r['overhead_vs_audit_%']:.1f}%, hash {r.us_hash_per_tx:.1f} µs/transaksi")
print(f"RQ4  Verifikasi penuh n={int(big.n):,}: {big.verifikasi_ms/1000:.2f} s ({big.us_verif_per_entri:.2f} µs/entri, "
      f"{big.throughput_verif_entri_s:,.0f} entri/s); inkremental {inc_summary.loc['mean','speedup']:.0f}x lebih cepat.")
print(f"RQ5  Log audit konvensional dapat diubah diam-diam; dengan rantai hash setiap perubahan pada {len(SCENARIOS)} skenario "
      f"insider terdeteksi dan terlokalisasi, dan jangkar eksternal menutup celah penulisan ulang total.")

RQ1  Implementasi: paket PL/SQL ABC_CDIVF (DBMS_CRYPTO SHA-256), serialisasi kanonik tervalidasi silang dengan Python.
RQ2  TDR = 100.0% (270 serangan, 9 jenis), FPR = 0.0% (30 uji bersih); penulisan ulang rantai penuh hanya terdeteksi oleh jangkar eksternal.
RQ3  n=    1,000: overhead CRUD 638.5% (11.4 -> 84.5 µs/operasi), vs audit trigger 85.7%, hash 16.6 µs/transaksi
RQ3  n=   10,000: overhead CRUD 490.4% (16.6 -> 97.9 µs/operasi), vs audit trigger 149.8%, hash 16.3 µs/transaksi
RQ3  n=  100,000: overhead CRUD 347.4% (20.0 -> 89.4 µs/operasi), vs audit trigger 113.8%, hash 16.2 µs/transaksi
RQ3  n=1,000,000: overhead CRUD 323.0% (24.1 -> 101.8 µs/operasi), vs audit trigger 96.3%, hash 16.6 µs/transaksi
RQ4  Verifikasi penuh n=1,000,000: 19.49 s (17.56 µs/entri, 56,947 entri/s); inkremental 103x lebih cepat.
RQ5  Log audit konvensional dapat diubah diam-diam; dengan rantai hash setiap perubahan pada 12 skenario insider terdeteksi dan terlokalisasi, dan jangkar eksternal menutup celah